# Cartpole (Inverted Pendulum)

**Companion notebook for [04_cartpole.md](../../docs/guides/getting_familiar/04_cartpole.md)**

The cartpole is a classic control benchmark: a cart on a frictionless track with an inverted pendulum (pole) attached at a pivot. The goal is to balance the pole upright by applying horizontal forces to the cart.

**Key features:**
- 4 state variables: `[x, x_dot, theta, theta_dot]` (cart position/velocity, pole angle/angular velocity)
- 1 control input: `F` (horizontal force on cart)
- Coupled nonlinear dynamics (Barto, Sutton, Anderson 1983)
- Unstable upright equilibrium (requires active control to balance)
- Stable hanging equilibrium (natural resting position)

This notebook covers:
1. Creating and exploring the cartpole model
2. Free dynamics from small perturbations
3. Phase portraits and equilibria analysis
4. Energy concepts
5. **LQR stabilization** - designing a controller to balance the pole

## Setup and Imports

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp

from fmd.simulator import (
    Cartpole,
    simulate,
    ConstantControl,
    ZeroControl,
    CART_X,
    CART_X_DOT,
    CART_THETA,
    CART_THETA_DOT,
    linearize,
)
from fmd.simulator.params import (
    CartpoleParams,
    CARTPOLE_CLASSIC,
    CARTPOLE_HEAVY_POLE,
    CARTPOLE_LONG_POLE,
)
from fmd.simulator.lqr import LQRController

# Create the cartpole model with classic parameters (OpenAI Gym style)
cartpole = Cartpole(CARTPOLE_CLASSIC)

print("Cartpole parameters (CARTPOLE_CLASSIC preset):")
print(f"  mass_cart = {cartpole.mass_cart} kg")
print(f"  mass_pole = {cartpole.mass_pole} kg")
print(f"  pole_length = {cartpole.pole_length} m (half-length to CoM)")
print(f"  total_mass = {cartpole.total_mass} kg")
print(f"  g = {cartpole.g} m/s^2")
print(f"\nState vector: {cartpole.state_names}")
print(f"Control vector: {cartpole.control_names}")

## Part 1: Understanding the Dynamics

### Model Equations

The cartpole uses the Barto, Sutton, Anderson (1983) equations of motion:

**State vector:** $[x, \dot{x}, \theta, \dot{\theta}]$
- $x$: cart position (m)
- $\dot{x}$: cart velocity (m/s)
- $\theta$: pole angle from vertical (rad), positive = clockwise
- $\dot{\theta}$: pole angular velocity (rad/s)

**Dynamics:**
$$\ddot{\theta} = \frac{g \sin\theta + \cos\theta \left( \frac{-F - m_p l \dot{\theta}^2 \sin\theta}{m_c + m_p} \right)}{l \left( \frac{4}{3} - \frac{m_p \cos^2\theta}{m_c + m_p} \right)}$$

$$\ddot{x} = \frac{F + m_p l \left( \dot{\theta}^2 \sin\theta - \ddot{\theta} \cos\theta \right)}{m_c + m_p}$$

Notice the coupling: the pole's angular acceleration depends on the cart's force, and the cart's acceleration depends on the pole's motion.

### Equilibrium States

The cartpole has two equilibrium configurations:
- **Upright** ($\theta = 0$): Pole balanced vertically above the cart - **unstable**
- **Hanging** ($\theta = \pi$): Pole hanging below the cart - **stable**

In [ ]:
# Examine equilibrium states
upright = cartpole.upright_state()
hanging = cartpole.hanging_state()

print("Upright equilibrium (unstable):")
print(f"  state = {upright}")
print(f"  theta = {float(upright[CART_THETA]):.4f} rad = {np.degrees(float(upright[CART_THETA])):.1f} deg")

print("\nHanging equilibrium (stable):")
print(f"  state = {hanging}")
print(f"  theta = {float(hanging[CART_THETA]):.4f} rad = {np.degrees(float(hanging[CART_THETA])):.1f} deg")

# Verify zero derivative at equilibria (with no control)
print("\nState derivatives at equilibria (should be ~0):")
deriv_upright = cartpole.forward_dynamics(upright, jnp.array([0.0]))
deriv_hanging = cartpole.forward_dynamics(hanging, jnp.array([0.0]))
print(f"  d/dt(upright) = {deriv_upright}")
print(f"  d/dt(hanging) = {deriv_hanging}")

## Part 2: Free Dynamics (No Control)

Let's simulate the cartpole from a small perturbation near the upright position to see the instability.

In [ ]:
# Simulate free fall from a small initial tilt
theta0 = 0.1  # Initial angle (rad) ~ 5.7 degrees
initial_state = jnp.array([0.0, 0.0, theta0, 0.0])

# Simulate with no control
result = simulate(cartpole, initial_state, dt=0.001, duration=3.0)

times = np.asarray(result.times)
states = np.asarray(result.states)

# Plot the state trajectories
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Pole angle
ax = axes[0, 0]
ax.plot(times, np.degrees(states[:, CART_THETA]), 'b-', linewidth=2)
ax.axhline(0, color='r', linestyle=':', alpha=0.5, label='Upright (0 deg)')
ax.axhline(180, color='g', linestyle=':', alpha=0.5, label='Hanging (180 deg)')
ax.axhline(-180, color='g', linestyle=':', alpha=0.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Pole angle (deg)')
ax.set_title(f'Pole Angle (initial tilt = {np.degrees(theta0):.1f} deg)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

# Angular velocity
ax = axes[0, 1]
ax.plot(times, np.degrees(states[:, CART_THETA_DOT]), 'b-', linewidth=2)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Angular velocity (deg/s)')
ax.set_title('Pole Angular Velocity')
ax.grid(True, alpha=0.3)

# Cart position
ax = axes[1, 0]
ax.plot(times, states[:, CART_X], 'b-', linewidth=2)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Cart position (m)')
ax.set_title('Cart Position (moves as pole falls)')
ax.grid(True, alpha=0.3)

# Cart velocity
ax = axes[1, 1]
ax.plot(times, states[:, CART_X_DOT], 'b-', linewidth=2)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Cart velocity (m/s)')
ax.set_title('Cart Velocity')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Initial angle: {np.degrees(theta0):.1f} deg")
print(f"Final angle: {np.degrees(states[-1, CART_THETA]):.1f} deg")
print(f"\nThe pole falls away from the upright position - this is the instability!")

## Part 3: Phase Portrait

The phase portrait shows trajectories in $(\theta, \dot{\theta})$ space. This reveals the system's qualitative behavior:
- **Saddle point** at $(0, 0)$: upright equilibrium (unstable)
- **Center** at $(\pm\pi, 0)$: hanging equilibrium (stable, oscillatory)

In [ ]:
# Phase portrait: simulate from multiple initial conditions
fig, ax = plt.subplots(figsize=(10, 8))

# Initial angles to simulate from
initial_angles = [0.05, 0.1, 0.3, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
colors = plt.cm.viridis(np.linspace(0, 1, len(initial_angles)))

for theta0, color in zip(initial_angles, colors):
    # Start from rest at each angle
    state0 = jnp.array([0.0, 0.0, theta0, 0.0])
    result = simulate(cartpole, state0, dt=0.001, duration=8.0)
    states = np.asarray(result.states)
    
    ax.plot(np.degrees(states[:, CART_THETA]),
            np.degrees(states[:, CART_THETA_DOT]),
            color=color, linewidth=1.5, alpha=0.7,
            label=f'theta0 = {np.degrees(theta0):.0f} deg')
    # Mark starting point
    ax.plot(np.degrees(theta0), 0, 'o', color=color, markersize=6)

# Mark equilibrium points
ax.plot(0, 0, 'rs', markersize=12, label='Upright (unstable)', zorder=5)
ax.plot(180, 0, 'g^', markersize=12, label='Hanging (stable)', zorder=5)
ax.plot(-180, 0, 'g^', markersize=12, zorder=5)

ax.set_xlabel('Pole angle theta (deg)', fontsize=12)
ax.set_ylabel('Angular velocity theta_dot (deg/s)', fontsize=12)
ax.set_title('Phase Portrait (cart free to move)', fontsize=14)
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(-200, 200)

plt.tight_layout()
plt.show()

print("Observations:")
print("- Trajectories near theta=0 diverge quickly (unstable saddle point)")
print("- Trajectories near theta=180 deg oscillate around the hanging position")
print("- The separatrix divides behaviors: some go over the top, others swing back")

## Part 4: Energy Analysis

Total mechanical energy is conserved (with no control input):

$$E = KE_{cart} + KE_{pole} + PE_{pole}$$

The potential energy is highest at the upright position and lowest at hanging.

In [ ]:
# Energy at equilibrium states
E_upright = float(cartpole.energy(cartpole.upright_state()))
E_hanging = float(cartpole.energy(cartpole.hanging_state()))

print("Energy at equilibrium states:")
print(f"  Upright (theta=0): E = {E_upright:.4f} J")
print(f"  Hanging (theta=pi): E = {E_hanging:.4f} J")
print(f"  Energy difference: {E_upright - E_hanging:.4f} J")

# Energy landscape vs theta (at rest)
theta_range = np.linspace(-np.pi, np.pi, 100)
energies_landscape = [float(cartpole.energy(jnp.array([0.0, 0.0, theta, 0.0]))) for theta in theta_range]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Energy landscape
ax = axes[0]
ax.plot(np.degrees(theta_range), energies_landscape, 'b-', linewidth=2)
ax.axhline(E_upright, color='r', linestyle='--', alpha=0.7, label=f'Upright E = {E_upright:.3f} J')
ax.axhline(E_hanging, color='g', linestyle='--', alpha=0.7, label=f'Hanging E = {E_hanging:.3f} J')
ax.axvline(0, color='r', linestyle=':', alpha=0.5)
ax.axvline(180, color='g', linestyle=':', alpha=0.5)
ax.axvline(-180, color='g', linestyle=':', alpha=0.5)
ax.set_xlabel('Pole angle theta (deg)')
ax.set_ylabel('Total energy (J)')
ax.set_title('Energy Landscape (cart at rest)')
ax.legend()
ax.grid(True, alpha=0.3)

# Energy conservation during simulation
theta0 = 0.3
state0 = jnp.array([0.0, 0.0, theta0, 0.0])
result = simulate(cartpole, state0, dt=0.0001, duration=5.0)

times = np.asarray(result.times)
states = np.asarray(result.states)
energies = np.array([float(cartpole.energy(states[i])) for i in range(len(times))])

ax = axes[1]
ax.plot(times, energies, 'b-', linewidth=2)
ax.axhline(energies[0], color='r', linestyle='--', alpha=0.7, label=f'Initial E = {energies[0]:.6f} J')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Total energy (J)')
ax.set_title('Energy Conservation (no control)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

energy_drift = abs(energies[-1] - energies[0])
print(f"\nEnergy conservation check (dt=0.0001s):")
print(f"  Initial energy: {energies[0]:.8f} J")
print(f"  Final energy: {energies[-1]:.8f} J")
print(f"  Energy drift: {energy_drift:.2e} J ({100*energy_drift/energies[0]:.4f}%)")

## Part 5: LQR Stabilization

Now we design a controller to **balance the pole upright**. We use Linear Quadratic Regulator (LQR), which:

1. **Linearizes** the dynamics around the upright equilibrium
2. **Solves** for optimal feedback gains that minimize a quadratic cost
3. **Applies** the control law: $u = -K(x - x_{ref})$

### Step 1: Linearize Around Upright Equilibrium

In [ ]:
# Linearize around the upright equilibrium
x_eq = cartpole.upright_state()  # [0, 0, 0, 0]
u_eq = jnp.array([0.0])  # No force at equilibrium

A, B = linearize(cartpole, x_eq, u_eq)

print("Linearized system: dx/dt = A*x + B*u")
print("\nA matrix (state dynamics):")
print(np.array(A))
print("\nB matrix (control influence):")
print(np.array(B))

# Check eigenvalues of A (should have positive real parts - unstable)
eigenvalues_A = np.linalg.eigvals(np.array(A))
print("\nOpen-loop eigenvalues of A:")
for i, eig in enumerate(eigenvalues_A):
    stability = "UNSTABLE" if eig.real > 0 else "stable" if eig.real < 0 else "marginal"
    print(f"  lambda_{i+1} = {eig.real:+.4f} {'+' if eig.imag >= 0 else ''}{eig.imag:.4f}j  ({stability})")

print("\nNote: Positive real eigenvalue confirms the upright position is unstable!")

### Step 2: Design LQR Controller

LQR minimizes the infinite-horizon cost:

$$J = \int_0^\infty (x^T Q x + u^T R u) \, dt$$

- **Q** penalizes state deviations (larger Q = tighter tracking)
- **R** penalizes control effort (larger R = gentler control)

In [ ]:
# Design LQR controller
# Q: state cost (penalize position, velocity, angle, angular velocity)
Q = jnp.diag(jnp.array([
    1.0,   # x: cart position
    1.0,   # x_dot: cart velocity
    10.0,  # theta: pole angle (higher weight = keep pole more upright)
    10.0   # theta_dot: angular velocity
]))

# R: control cost (penalize force magnitude)
R = jnp.array([[0.1]])

print("Cost matrices:")
print(f"Q = diag{list(np.diag(Q))}")
print(f"R = {float(R[0,0])}")

# Create LQR controller using fomodynamics's built-in method
controller = LQRController.from_linearization(
    cartpole, x_eq, u_eq, Q, R
)

print("\nLQR feedback gain K:")
print(f"  K = {np.array(controller.K)}")
print("\nControl law: u = -K @ (x - x_ref)")
print(f"  u = -({controller.K[0,0]:.3f}*x + {controller.K[0,1]:.3f}*x_dot + {controller.K[0,2]:.3f}*theta + {controller.K[0,3]:.3f}*theta_dot)")

In [ ]:
# Check closed-loop stability
K = np.array(controller.K)
A_cl = np.array(A) - np.array(B) @ K  # Closed-loop dynamics

eigenvalues_cl = np.linalg.eigvals(A_cl)
print("Closed-loop eigenvalues (A - B*K):")
for i, eig in enumerate(eigenvalues_cl):
    stability = "UNSTABLE" if eig.real > 0 else "stable" if eig.real < 0 else "marginal"
    print(f"  lambda_{i+1} = {eig.real:+.4f} {'+' if eig.imag >= 0 else ''}{eig.imag:.4f}j  ({stability})")

all_stable = all(e.real < 0 for e in eigenvalues_cl)
print(f"\nAll eigenvalues have negative real parts: {all_stable}")
if all_stable:
    print("The closed-loop system is STABLE - LQR controller will balance the pole!")

### Step 3: Simulate Closed-Loop Response

Now we test the LQR controller by starting from a perturbed initial condition and simulating the closed-loop system.

In [ ]:
# Simulate with LQR controller from a perturbed initial condition
theta0_deg = 15.0  # Initial angle in degrees
theta0 = np.radians(theta0_deg)
x0 = jnp.array([0.0, 0.0, theta0, 0.0])  # Start displaced

# Simulate with controller
result_lqr = simulate(cartpole, x0, dt=0.001, duration=5.0, control=controller)

times_lqr = np.asarray(result_lqr.times)
states_lqr = np.asarray(result_lqr.states)

# Also compute the control effort over time
controls_lqr = np.array([float(controller(t, states_lqr[i])[0]) for i, t in enumerate(times_lqr)])

# Plot results
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Pole angle
ax = axes[0, 0]
ax.plot(times_lqr, np.degrees(states_lqr[:, CART_THETA]), 'b-', linewidth=2)
ax.axhline(0, color='g', linestyle='--', alpha=0.7, label='Target (upright)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Pole angle (deg)')
ax.set_title(f'Pole Angle (initial = {theta0_deg:.0f} deg)')
ax.legend()
ax.grid(True, alpha=0.3)

# Cart position
ax = axes[0, 1]
ax.plot(times_lqr, states_lqr[:, CART_X], 'b-', linewidth=2)
ax.axhline(0, color='g', linestyle='--', alpha=0.7, label='Target')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Cart position (m)')
ax.set_title('Cart Position')
ax.legend()
ax.grid(True, alpha=0.3)

# Control effort
ax = axes[1, 0]
ax.plot(times_lqr, controls_lqr, 'r-', linewidth=2)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Force (N)')
ax.set_title('Control Effort (Force on Cart)')
ax.grid(True, alpha=0.3)

# Phase portrait showing convergence
ax = axes[1, 1]
ax.plot(np.degrees(states_lqr[:, CART_THETA]), 
        np.degrees(states_lqr[:, CART_THETA_DOT]), 'b-', linewidth=2)
ax.plot(np.degrees(theta0), 0, 'go', markersize=10, label='Start')
ax.plot(0, 0, 'r*', markersize=15, label='Target (upright)')
ax.set_xlabel('Pole angle (deg)')
ax.set_ylabel('Angular velocity (deg/s)')
ax.set_title('Phase Portrait (converging to upright)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Initial state: x={x0[0]:.2f}m, theta={np.degrees(x0[2]):.1f}deg")
print(f"Final state: x={states_lqr[-1, CART_X]:.4f}m, theta={np.degrees(states_lqr[-1, CART_THETA]):.4f}deg")
print(f"Max control force: {np.max(np.abs(controls_lqr)):.2f} N")
print("\nThe LQR controller successfully balances the pole!")

### Step 4: Compare Controlled vs Uncontrolled

In [ ]:
# Compare controlled vs uncontrolled from same initial condition
theta0_deg = 10.0
theta0 = np.radians(theta0_deg)
x0 = jnp.array([0.0, 0.0, theta0, 0.0])

# Uncontrolled simulation
result_open = simulate(cartpole, x0, dt=0.001, duration=3.0)
times_open = np.asarray(result_open.times)
states_open = np.asarray(result_open.states)

# Controlled simulation
result_closed = simulate(cartpole, x0, dt=0.001, duration=3.0, control=controller)
times_closed = np.asarray(result_closed.times)
states_closed = np.asarray(result_closed.states)

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pole angle comparison
ax = axes[0]
ax.plot(times_open, np.degrees(states_open[:, CART_THETA]), 'r-', linewidth=2, label='No control (falls)')
ax.plot(times_closed, np.degrees(states_closed[:, CART_THETA]), 'b-', linewidth=2, label='LQR control (stabilizes)')
ax.axhline(0, color='g', linestyle='--', alpha=0.7, label='Target (upright)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Pole angle (deg)')
ax.set_title(f'Pole Angle: Controlled vs Uncontrolled (initial = {theta0_deg:.0f} deg)')
ax.legend()
ax.grid(True, alpha=0.3)

# Cart position comparison
ax = axes[1]
ax.plot(times_open, states_open[:, CART_X], 'r-', linewidth=2, label='No control')
ax.plot(times_closed, states_closed[:, CART_X], 'b-', linewidth=2, label='LQR control')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Cart position (m)')
ax.set_title('Cart Position: Controlled vs Uncontrolled')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Without control: The pole falls and the cart drifts away")
print("With LQR control: The pole is stabilized upright and the cart returns to center")

### Bonus: Testing Larger Initial Angles

LQR is designed using a **linear approximation** around the equilibrium. Let's see how well it works for larger initial angles.

In [ ]:
# Test LQR from various initial angles
initial_angles_deg = [5, 15, 30, 45, 60]
colors = plt.cm.plasma(np.linspace(0.2, 0.8, len(initial_angles_deg)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for theta0_deg, color in zip(initial_angles_deg, colors):
    theta0 = np.radians(theta0_deg)
    x0 = jnp.array([0.0, 0.0, theta0, 0.0])
    
    result = simulate(cartpole, x0, dt=0.001, duration=5.0, control=controller)
    times = np.asarray(result.times)
    states = np.asarray(result.states)
    
    axes[0].plot(times, np.degrees(states[:, CART_THETA]), color=color, 
                 linewidth=2, label=f'{theta0_deg} deg')
    axes[1].plot(times, states[:, CART_X], color=color, linewidth=2, label=f'{theta0_deg} deg')

axes[0].axhline(0, color='g', linestyle='--', alpha=0.7)
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Pole angle (deg)')
axes[0].set_title('LQR Stabilization from Various Initial Angles')
axes[0].legend(title='Initial angle')
axes[0].grid(True, alpha=0.3)

axes[1].axhline(0, color='g', linestyle='--', alpha=0.7)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Cart position (m)')
axes[1].set_title('Cart Position for Various Initial Angles')
axes[1].legend(title='Initial angle')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Observations:")
print("- LQR works well for small to moderate initial angles")
print("- Larger angles require more cart motion to recover")
print("- Very large angles (>60 deg) may cause the linear approximation to break down")

---

## Bonus: Pole Tip Trajectory

Visualize the motion of the pole tip in Cartesian space as the pole swings. This gives an intuitive view of the pendulum's behavior beyond just the angle plots.

In [ ]:
# Pole tip trajectory during fall
theta0 = 0.3
state0 = jnp.array([0.0, 0.0, theta0, 0.0])
result = simulate(cartpole, state0, dt=0.001, duration=5.0)

times = np.asarray(result.times)
states = np.asarray(result.states)

# Compute pole tip positions
x_tips = []
y_tips = []
for i in range(len(times)):
    x_tip, y_tip = cartpole.pole_tip_position(states[i])
    x_tips.append(float(x_tip))
    y_tips.append(float(y_tip))

x_tips = np.array(x_tips)
y_tips = np.array(y_tips)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pole tip trajectory
ax = axes[0]
scatter = ax.scatter(x_tips, y_tips, c=times, cmap='viridis', s=1)
ax.plot(x_tips[0], y_tips[0], 'go', markersize=10, label='Start')
ax.plot(x_tips[-1], y_tips[-1], 'ro', markersize=10, label='End')
ax.axhline(0, color='k', linestyle='-', alpha=0.3)  # Track level
plt.colorbar(scatter, ax=ax, label='Time (s)')
ax.set_xlabel('X position (m)')
ax.set_ylabel('Y position (m)')
ax.set_title('Pole Tip Trajectory')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

# Animation snapshots
ax = axes[1]
snapshot_times = [0, 0.5, 1.0, 1.5, 2.0]
colors = plt.cm.Blues(np.linspace(0.3, 1.0, len(snapshot_times)))

for t_snap, color in zip(snapshot_times, colors):
    idx = np.argmin(np.abs(times - t_snap))
    x_cart = states[idx, CART_X]
    x_tip = x_tips[idx]
    y_tip = y_tips[idx]
    
    # Draw cart (simple rectangle)
    ax.plot([x_cart - 0.1, x_cart + 0.1], [0, 0], '-', color=color, linewidth=8)
    # Draw pole
    ax.plot([x_cart, x_tip], [0, y_tip], '-', color=color, linewidth=3)
    ax.plot(x_tip, y_tip, 'o', color=color, markersize=8)

ax.axhline(0, color='k', linestyle='-', linewidth=2)  # Track
ax.set_xlabel('X position (m)')
ax.set_ylabel('Y position (m)')
ax.set_title(f'Snapshots at t = {snapshot_times}')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
ax.set_ylim(-0.5, 1.5)

plt.tight_layout()
plt.show()

## Bonus: Comparing Parameter Presets

fomodynamics provides several cartpole presets with different physical parameters. Let's compare how they behave when released from the same initial angle.

In [ ]:
# Compare presets: free fall from same initial angle
preset_map = {
    'CARTPOLE_CLASSIC': CARTPOLE_CLASSIC,
    'CARTPOLE_HEAVY_POLE': CARTPOLE_HEAVY_POLE,
    'CARTPOLE_LONG_POLE': CARTPOLE_LONG_POLE,
}

theta0 = 0.1  # rad
duration = 2.0
dt = 0.001

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for name, params in preset_map.items():
    cp = Cartpole(params)
    state0 = jnp.array([0.0, 0.0, theta0, 0.0])
    result = simulate(cp, state0, dt=dt, duration=duration)
    
    times = np.asarray(result.times)
    states = np.asarray(result.states)
    
    # Pole angle
    axes[0].plot(times, np.degrees(states[:, CART_THETA]), linewidth=2, label=name)
    # Angular velocity
    axes[1].plot(times, np.degrees(states[:, CART_THETA_DOT]), linewidth=2, label=name)
    # Cart position
    axes[2].plot(times, states[:, CART_X], linewidth=2, label=name)

axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Pole angle (deg)')
axes[0].set_title('Pole Angle vs Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Angular velocity (deg/s)')
axes[1].set_title('Angular Velocity vs Time')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].set_xlabel('Time (s)')
axes[2].set_ylabel('Cart position (m)')
axes[2].set_title('Cart Position vs Time')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print comparison table
print("\nPreset comparison:")
print(f"{'Preset':<20} {'Pole length':>12} {'Period':>10} {'Mass ratio':>12}")
print("-" * 56)
for name, params in preset_map.items():
    cp = Cartpole(params)
    print(f"{name:<20} {cp.pole_length:>12.2f} m {float(cp.linearized_period()):>10.3f} s {cp.mass_pole/cp.mass_cart:>12.3f}")

print("\nKey insight: Longer poles have longer periods and are easier to balance!")

## Bonus: Autodiff Through Simulation

One of JAX's superpowers is automatic differentiation. We can compute gradients of any scalar loss function with respect to initial conditions or model parameters - even when the loss involves running an entire simulation.

In [ ]:
# Autodiff: compute gradient of final angle w.r.t. initial angle
def final_angle_loss(theta0):
    """Loss = squared final angle after simulation."""
    state0 = jnp.array([0.0, 0.0, theta0, 0.0])
    result = simulate(cartpole, state0, dt=0.01, duration=0.5)
    return result.states[-1, CART_THETA] ** 2

# Compute gradient at a test point
theta0_test = 0.1
loss = final_angle_loss(theta0_test)
grad = jax.grad(final_angle_loss)(theta0_test)

print("Gradient of loss w.r.t. initial angle:")
print(f"  theta0 = {theta0_test} rad")
print(f"  Loss (final angle squared) = {float(loss):.6f}")
print(f"  d(loss)/d(theta0) = {float(grad):.6f}")

# We can also differentiate w.r.t. model parameters!
def loss_wrt_length(pole_length):
    """Loss as a function of pole length."""
    cp = Cartpole.from_values(
        mass_cart=1.0,
        mass_pole=0.1,
        pole_length=pole_length,
    )
    state0 = jnp.array([0.0, 0.0, 0.1, 0.0])
    result = simulate(cp, state0, dt=0.01, duration=0.5)
    return result.states[-1, CART_THETA] ** 2

length_test = 0.5
loss = loss_wrt_length(length_test)
grad = jax.grad(loss_wrt_length)(length_test)

print("\nGradient of loss w.r.t. pole length:")
print(f"  pole_length = {length_test} m")
print(f"  Loss = {float(loss):.6f}")
print(f"  d(loss)/d(pole_length) = {float(grad):.6f}")
print("\nNegative gradient means longer poles lead to smaller final angles (easier to balance)!")

## Summary

In this notebook, we explored the cartpole (inverted pendulum) model:

**Dynamics:**
- 4-state system: cart position/velocity and pole angle/angular velocity
- Coupled nonlinear dynamics
- Unstable upright equilibrium, stable hanging equilibrium

**Analysis:**
- Phase portraits reveal saddle-point (upright) and center (hanging) equilibria
- Energy is conserved in free dynamics
- Open-loop eigenvalues confirm instability at upright position

**LQR Control:**
- Linearized the system around the upright equilibrium
- Designed LQR controller with state cost Q and control cost R
- Verified closed-loop stability (all eigenvalues have negative real parts)
- Successfully balanced the pole from various initial perturbations

**Key takeaways:**
- LQR is a powerful tool for stabilizing unstable systems
- The Q/R ratio trades off tracking performance vs control effort
- Linear controllers work well near the equilibrium, but may struggle far from it

## Next Steps

- **[06 - Planar Quadrotor](06_planar_quadrotor.md)**: 2D flight dynamics with thrust control
- **[Control Guide](../../docs/control_guide.md)**: LQR design, eigenvalue analysis, and timestep selection
